In [21]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import r2_score
from xgboost import XGBRegressor
from sklearn.model_selection import KFold,cross_val_score
from sklearn.compose import make_column_selector, make_column_transformer
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_auc_score, log_loss

In [22]:
df = pd.read_csv("cirrhosis.csv")
df.head()

,ID,N_Days,Status,Drug,Age,Sex,Ascites,Hepatomegaly,Spiders,Edema,Bilirubin,Cholesterol,Albumin,Copper,Alk_Phos,SGOT,Tryglicerides,Platelets,Prothrombin,Stage
0,1,400,D,D-penicillamine,21464,F,Y,Y,Y,Y,14.5,261.0,2.60,156.0,1718.0,137.95,172.0,190.0,12.2,4.0
1,2,4500,C,D-penicillamine,20617,F,N,Y,Y,N,1.1,302.0,4.14,54.0,7394.8,113.52,88.0,221.0,10.6,3.0
2,3,1012,D,D-penicillamine,25594,M,N,N,N,S,1.4,176.0,3.48,210.0,516.0,96.10,55.0,151.0,12.0,4.0
3,4,1925,D,D-penicillamine,19994,F,N,Y,Y,S,1.8,244.0,2.54,64.0,6121.8,60.63,92.0,183.0,10.3,4.0
4,5,1504,CL,Placebo,13918,F,N,Y,Y,N,3.4,279.0,3.53,143.0,671.0,113.15,72.0,136.0,10.9,3.0


In [23]:
x,y = df.drop("Status",axis=1), df["Status"]
df.isnull().sum().sum()

x = x.replace("NaNN",np.nan)
x["Cholesterol"] = x["Cholesterol"].astype("float")
x["Copper"] = x["Copper"].astype("float")
x["Tryglicerides"] = x["Tryglicerides"].astype("float")
x["Platelets"] = x["Platelets"].astype("float")

sci_scaler, mm_scaler = StandardScaler(),MinMaxScaler()
mean_imputer = SimpleImputer()
str_impute = SimpleImputer(strategy="most_frequent")
median_imputer = SimpleImputer(strategy="median")
lb = LabelEncoder()
le_y = lb.fit_transform(y)
knn = KNeighborsClassifier()

In [24]:
ohc = OneHotEncoder(sparse_output=False, drop="first").set_output(transform="pandas")
ct_ohc = make_column_transformer((ohc,make_column_selector(dtype_include=object)),
                             ("passthrough", make_column_selector(dtype_include=['int64','float64'])),
                             verbose_feature_names_out=False).set_output(transform="pandas")

ct_str = make_column_transformer((str_impute,make_column_selector(dtype_include=object)),
                             (mean_imputer, make_column_selector(dtype_include=['int64','float64'])),
                             verbose_feature_names_out=False).set_output(transform="pandas")


pipe = Pipeline([("IMP",ct_str),("CT",ct_ohc),("SCL",sci_scaler),("KNN",knn)])
pipe.get_params()

{'memory': None,
 'steps': [('IMP',
   ColumnTransformer(transformers=[('simpleimputer-1',
                                    SimpleImputer(strategy='most_frequent'),
                                    <sklearn.compose._column_transformer.make_column_selector object at 0x0000015FC40D2120>),
                                   ('simpleimputer-2', SimpleImputer(),
                                    <sklearn.compose._column_transformer.make_column_selector object at 0x0000015FC43DDF40>)],
                     verbose_feature_names_out=False)),
  ('CT',
   ColumnTransformer(transformers=[('onehotencoder',
                                    OneHotEncoder(drop='first',
                                                  sparse_output=False),
                                    <sklearn.compose._column_transformer.make_column_selector object at 0x0000015FC6213EC0>),
                                   ('passthrough', 'passthrough',
                                    <sklearn.compose._column_

In [18]:
df.columns

Index(['ID', 'N_Days', 'Status', 'Drug', 'Age', 'Sex', 'Ascites',
       'Hepatomegaly', 'Spiders', 'Edema', 'Bilirubin', 'Cholesterol',
       'Albumin', 'Copper', 'Alk_Phos', 'SGOT', 'Tryglicerides', 'Platelets',
       'Prothrombin', 'Stage'],
      dtype='str')

In [13]:

imputer = SimpleImputer()
# std_scaler = StandardScaler()
mm_scaler = MinMaxScaler()
knn =KNeighborsRegressor()
pipe = Pipeline([("IMP",imputer),("CT",ct),("SCL",mm_scaler),("KNN",knn)])

In [14]:
pipe.get_params()

{'memory': None,
 'steps': [('IMP', SimpleImputer()),
  ('CT',
   ColumnTransformer(transformers=[('onehotencoder',
                                    OneHotEncoder(drop='first',
                                                  sparse_output=False),
                                    <sklearn.compose._column_transformer.make_column_selector object at 0x0000015FC3ECE060>),
                                   ('passthrough', 'passthrough',
                                    <sklearn.compose._column_transformer.make_column_selector object at 0x0000015FC3D96630>)],
                     verbose_feature_names_out=False)),
  ('SCL', MinMaxScaler()),
  ('KNN', KNeighborsRegressor())],
 'transform_input': None,
 'verbose': False,
 'IMP': SimpleImputer(),
 'CT': ColumnTransformer(transformers=[('onehotencoder',
                                  OneHotEncoder(drop='first',
                                                sparse_output=False),
                                  <sklearn.compose._

In [15]:
params = {"IMP__strategy":['mean','median'],
          "SCL":[mm_scaler,None],
          "KNN__n_neighbors": np.arange(1,11)}
kfold = KFold(n_splits=5, shuffle=True, random_state=26)

gcv = GridSearchCV(pipe, param_grid=params, cv=kfold, n_jobs=-1)
gcv.fit(X,le_y)
print("Best params : ", gcv.best_params_)
print("Best score : ", gcv.best_score_)

ValueError: 
All the 200 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
100 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\PGCP-AI.STUDENTSDC\AppData\Roaming\Python\Python312\site-packages\sklearn\model_selection\_validation.py", line 855, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\PGCP-AI.STUDENTSDC\AppData\Roaming\Python\Python312\site-packages\sklearn\base.py", line 1403, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\PGCP-AI.STUDENTSDC\AppData\Roaming\Python\Python312\site-packages\sklearn\pipeline.py", line 632, in fit
    Xt = self._fit(
         ^^^^^^^^^^
  File "C:\Users\PGCP-AI.STUDENTSDC\AppData\Roaming\Python